In [1]:
import numpy as np
import torch
from huggingface_hub import hf_hub_download
from PIL import Image
from torchvision.transforms import v2
from src.models.lrm_mesh import InstantMesh
from src.utils.camera_util import get_zero123plus_input_cameras
from src.utils.mesh_util import xatlas_uvmap
import trimesh
import nvdiffrast.torch as dr
import cv2

device = torch.device("cuda")

In [2]:
model_ckpt_path = hf_hub_download(
    repo_id="TencentARC/InstantMesh",
    filename="instant_mesh_large.ckpt",
    repo_type="model"
)
model = InstantMesh(
    encoder_feat_dim=768,
    encoder_freeze=False,
    encoder_model_name="facebook/dino-vitb16",
    transformer_dim=1024,
    transformer_layers=16,
    transformer_heads=16,
    triplane_low_res=32,
    triplane_high_res=64,
    triplane_dim=80,
    rendering_samples_per_ray=128,
    grid_res=128,
    grid_scale=2.1
)
state_dict = torch.load(model_ckpt_path, map_location="cpu")["state_dict"]
state_dict = {
    k[14:]: v
    for k, v in state_dict.items()
    if k.startswith("lrm_generator.") and "source_camera" not in k
}
model.load_state_dict(state_dict, strict=True)
model = model.to(device)
model.init_flexicubes_geometry(device, fovy=30.0)
model = model.eval()

Some weights of ViTModel were not initialized from the model checkpoint at facebook/dino-vitb16 and are newly initialized: ['encoder.layer.0.adaLN_modulation.1.bias', 'encoder.layer.0.adaLN_modulation.1.weight', 'encoder.layer.1.adaLN_modulation.1.bias', 'encoder.layer.1.adaLN_modulation.1.weight', 'encoder.layer.10.adaLN_modulation.1.bias', 'encoder.layer.10.adaLN_modulation.1.weight', 'encoder.layer.11.adaLN_modulation.1.bias', 'encoder.layer.11.adaLN_modulation.1.weight', 'encoder.layer.2.adaLN_modulation.1.bias', 'encoder.layer.2.adaLN_modulation.1.weight', 'encoder.layer.3.adaLN_modulation.1.bias', 'encoder.layer.3.adaLN_modulation.1.weight', 'encoder.layer.4.adaLN_modulation.1.bias', 'encoder.layer.4.adaLN_modulation.1.weight', 'encoder.layer.5.adaLN_modulation.1.bias', 'encoder.layer.5.adaLN_modulation.1.weight', 'encoder.layer.6.adaLN_modulation.1.bias', 'encoder.layer.6.adaLN_modulation.1.weight', 'encoder.layer.7.adaLN_modulation.1.bias', 'encoder.layer.7.adaLN_modulation.1.w

In [3]:
input_cameras = get_zero123plus_input_cameras().to(device)

""" input_mesh = trimesh.load("tmp/mesh.obj")
input_mesh.vertices = input_mesh.vertices @ np.array([[1, 0, 0], [0, 1, 0], [0, 0, -1]])
input_mesh.faces = input_mesh.faces[:, [2, 1, 0]]

input_cameras_numpy = input_cameras.cpu().numpy()
extrinsics = input_cameras_numpy[0, :, :12].reshape(-1, 3, 4)
fov = 30.0

images = []
for i, extrinsic in enumerate(extrinsics):
    extrinsic4x4 = np.eye(4)
    extrinsic4x4[0, :] = extrinsic[1, :]
    extrinsic4x4[1, :] = extrinsic[2, :]
    extrinsic4x4[2, :] = extrinsic[0, :]
    camera = trimesh.scene.Camera(fov=(fov, fov))
    scene = trimesh.Scene(
        input_mesh,
        camera=camera,
        camera_transform=extrinsic4x4
    )
    image = scene.save_image(resolution=[1024, 1024], visible=False)
    with open(f"tmp/mesh_{i}.png", "wb") as f:
        f.write(image)

    image = Image.open(io.BytesIO(image)).convert("RGB")
    images.append(np.array(image, dtype=np.float32) / 255.0) """

""" images = []
image_paths = [f"tmp/output_image_{i}.png" for i in range(6)]
for image_path in image_paths:
    image = Image.open(image_path).convert("RGB")
    image = np.array(image, dtype=np.float32) / 255.0
    image = image.transpose(2, 0, 1)
    images.append(image)

images = np.stack(images, axis=0)
print(images.shape)
images_processed = torch.from_numpy(images)
images_processed = images_processed.unsqueeze(0).to(device)
images_processed = v2.functional.resize(
    images_processed, 
    320, 
    interpolation=3, 
    antialias=True
).clamp(0, 1) """

images_processed = torch.load("tmp/images_processed.pt").to(device)

/tmp/ipykernel_1967/1885027445.py:49: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  images_processed = torch.load("tmp/images_processed.pt").to(device)


In [4]:
with torch.no_grad():
    planes = model.forward_planes(images_processed, input_cameras)
    mesh_path = "tmp/mesh_post.obj"
    mesh_out = model.extract_mesh(
        planes,
        use_texture_map=False,
        texture_resolution=1024,
    )
    vertices_raw, faces_raw, vertex_colors = mesh_out
    vertices = vertices_raw.cpu().numpy()[:, [1, 2, 0]]
    vertices = vertices @ np.array([[1, 0, 0], [0, 1, 0], [0, 0, -1]])
    faces = faces_raw.cpu().numpy()[:, [2, 1, 0]]
    mesh = trimesh.Trimesh(
        vertices=vertices,
        faces=faces,
        vertex_colors=vertex_colors,
    )
    mesh.export(mesh_path, "obj")

In [5]:
with torch.no_grad():
    dr.RasterizeCudaContext(device=device)
    uvs, mesh_tex_idx, gb_pos, tex_hard_mask = xatlas_uvmap(
        model.geometry.renderer.ctx, vertices_raw, faces_raw, resolution=1024)
    tex_hard_mask = tex_hard_mask.float()

    tex_feat = model.get_texture_prediction(
        planes, [gb_pos], tex_hard_mask
    )
    background_feat = torch.zeros_like(tex_feat)
    img_feat = torch.lerp(background_feat, tex_feat, tex_hard_mask)
    texture_map = img_feat.permute(0, 3, 1, 2).squeeze(0)

    pointnp_px3 = vertices_raw.cpu().numpy()
    tcoords_px2 = uvs.cpu().numpy()
    facenp_fx3 = faces_raw.cpu().numpy()
    facetex_fx3 = mesh_tex_idx.cpu().numpy()
    texmap_hxwx3 = texture_map.permute(1, 2, 0).data.cpu().numpy()

    fol = "tmp"
    na = "mesh_post_tex"

    matname = f"{fol}/{na}.mtl"
    fid = open(matname, "w")
    fid.write("newmtl material_0\n")
    fid.write("Kd 1 1 1\n")
    fid.write("Ka 0 0 0\n")
    fid.write("Ks 0.4 0.4 0.4\n")
    fid.write("Ns 10\n")
    fid.write("illum 2\n")
    fid.write("map_Kd %s.png\n" % na)
    fid.close()

    fid = open(f"{fol}/{na}.obj", "w")
    fid.write("mtllib %s.mtl\n" % na)

    for pidx, p in enumerate(pointnp_px3):
        pp = p
        fid.write("v %f %f %f\n" % (pp[0], pp[1], pp[2]))

    for pidx, p in enumerate(tcoords_px2):
        pp = p
        fid.write("vt %f %f\n" % (pp[0], pp[1]))

    fid.write("usemtl material_0\n")
    for i, f in enumerate(facenp_fx3):
        f1 = f + 1
        f2 = facetex_fx3[i] + 1
        fid.write("f %d/%d %d/%d %d/%d\n" % (f1[0], f2[0], f1[1], f2[1], f1[2], f2[2]))
    fid.close()

    lo, hi = 0, 1
    img = np.asarray(texmap_hxwx3, dtype=np.float32)
    img = (img - lo) * (255 / (hi - lo))
    img = img.clip(0, 255)
    mask = np.sum(img.astype(np.float32), axis=-1, keepdims=True)
    mask = (mask <= 3.0).astype(np.float32)
    kernel = np.ones((3, 3), "uint8")
    dilate_img = cv2.dilate(img, kernel, iterations=1)
    img = img * (1 - mask) + dilate_img * mask
    img = img.clip(0, 255).astype(np.uint8)
    Image.fromarray(np.ascontiguousarray(img[::-1, :, :]), "RGB").save(
        f"{fol}/{na}.png"
    )